In [1]:
import os
import certifi
import requests
from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain.agents import create_agent
from langchain.tools import tool


C:\Users\THARINDU\AppData\Local\Temp\ipykernel_13288\1877827372.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools.tavily_search import TavilySearchResults


In [2]:
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHER_API_KEY = os.getenv('WEATHER_API_KEY')

In [3]:
search_tool = TavilySearchResults(
    max_results=2,)

C:\Users\THARINDU\AppData\Local\Temp\ipykernel_13288\1254000495.py:1: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  search_tool = TavilySearchResults(


In [4]:
@tool
def weather_tool(city: str) -> str:
    """Get the current weather for a city."""
    response = requests.get(
        "https://api.weatherstack.com/current",
        params={"access_key": WEATHER_API_KEY, "query": city},
        timeout=10,
    )
    response.raise_for_status()
    data = response.json()

    if "current" not in data:
        return f"Could not retrieve weather data for {city}."

    descriptions = data["current"].get("weather_descriptions", [])
    return f"The weather in {city} is {', '.join(descriptions)}."

In [5]:
result = search_tool.invoke({"query": "What is the capital of France?"})
result

[{'title': 'Paris - Wikipedia',
  'url': 'https://en.wikipedia.org/wiki/Paris',
  'content': 'Paris is the capital and largest city of France, with an estimated city population of 2.04 million in an area of 105.4 km2 (40.7 sq mi), as of January 2026( and a metropolitan population of 13.3 million (2023). Located on the Seine in the centre of the Île-de-France region, it is the largest metropolitan area and fourth-most populous city in the European Union (EU). Paris is nicknamed the "City of Light" mainly due to its leading role in the Age of Enlightenment. [...] Edit links\n\n Article\n Talk\n\n Read\n View source\n View history\n\nTools\n\nActions\n\n Read\n View source\n View history\n\nGeneral\n\n What links here\n Related changes\n Upload file\n Permanent link\n Page information\n Cite this page\n Get shortened URL\n Switch to legacy parser\n\nPrint/export\n\n Download as PDF\n Printable version\n\nIn other projects\n\n Abstract Wikipedia\n Wikimedia Commons\n Wikinews\n Wikiquote\n

In [6]:
llm = ChatGroq(
    model_name="openai/gpt-oss-120b",
    temperature=0,
    groq_api_key=GROQ_API_KEY,
)

In [7]:
response = llm.invoke("What is the capital of Austria?")
response

AIMessage(content='The capital of Austria is **Vienna**.', additional_kwargs={'reasoning_content': 'The user asks: "What is the capital of Austria?" Straightforward answer: Vienna. Provide answer.'}, response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 78, 'total_tokens': 119, 'completion_time': 0.086225841, 'completion_tokens_details': {'reasoning_tokens': 22}, 'prompt_time': 0.017342393, 'prompt_tokens_details': None, 'queue_time': 0.215921224, 'total_time': 0.103568234}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_93703442d9', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1069e-ec66-7de0-885c-4142ae7b078b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 78, 'output_tokens': 41, 'total_tokens': 119, 'output_token_details': {'reasoning': 22}})

In [8]:
tools = [search_tool, weather_tool]

In [9]:
agent = create_agent(
    model=llm,
    tools=tools,
)

In [10]:
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "find the current weather in Paris and the capital of Austria",
            }
        ]
    }
)

print(result["messages"][-1].content)

The current weather in Paris is **sunny**.  

The capital of Austria is **Vienna**.
